# 04-02 멀티헤드 어텐션

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 헤드 나누기: 모양 바꾸기만으로

In [ ]:
B, T, d_model, n_heads = 2, 5, 16, 4
d_head = d_model // n_heads  # 4

torch.manual_seed(0)
x = torch.randn(B, T, d_model)  # (2, 5, 16)
h = x.view(B, T, n_heads, d_head)  # (2, 5, 4, 4)
h = h.transpose(1, 2)  # (2, 4, 5, 4) = (batch, head, seq_len, d_head)
print(x.shape, "->", h.shape)

back = h.transpose(1, 2).reshape(B, T, d_model)  # (2, 5, 16)
print(torch.equal(back, x))

## 2. 멀티헤드 어텐션 직접 구현

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, T, d_head)

    def forward(self, x, mask=None):
        # x: (B, T, d_model), mask: (B, 1, 1, T) 또는 (1, 1, T, T), True인 곳만 봐요
        B, T, d_model = x.shape
        q = self.split(self.q_proj(x))  # (B, H, T, d_head)
        k = self.split(self.k_proj(x))  # (B, H, T, d_head)
        v = self.split(self.v_proj(x))  # (B, H, T, d_head)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        self.weights = F.softmax(scores, dim=-1)  # (B, H, T, T), 시각화용으로 저장
        out = self.weights @ v  # (B, H, T, d_head)
        out = out.transpose(1, 2).reshape(B, T, d_model)  # (B, T, d_model), 헤드 이어 붙이기
        return self.out_proj(out)  # (B, T, d_model)

mha = MultiHeadAttention(d_model, n_heads)
y = mha(x)
print(x.shape, "->", y.shape)
print(mha.weights.shape)

In [ ]:
fig, axes = plt.subplots(1, n_heads, figsize=(14, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(mha.weights[0, i].detach(), cmap="Oranges", vmin=0, vmax=1)
    ax.set_title(f"head {i}")
    ax.set_xlabel("key")
axes[0].set_ylabel("query")
plt.show()

## 3. 패딩 마스크

In [ ]:
lengths = torch.tensor([5, 3])  # 두 번째 문장은 진짜 토큰이 3개
pad_mask = torch.arange(T).unsqueeze(0) < lengths.unsqueeze(1)  # (B, T)
print(pad_mask)
mask = pad_mask[:, None, None, :]  # (B, 1, 1, T), 모든 헤드와 쿼리에 같은 마스크
y = mha(x, mask)
print(mha.weights[1, 0])  # 두 번째 문장, 0번 헤드: 마지막 두 열이 0

## 4. nn.MultiheadAttention과 비교하기

In [ ]:
torch_mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
with torch.no_grad():
    # PyTorch는 q, k, v 가중치를 하나로 이어 붙여 저장해요
    torch_mha.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight]))
    torch_mha.in_proj_bias.copy_(torch.cat([mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias]))
    torch_mha.out_proj.weight.copy_(mha.out_proj.weight)
    torch_mha.out_proj.bias.copy_(mha.out_proj.bias)

y_torch, _ = torch_mha(x, x, x, key_padding_mask=~pad_mask)  # PyTorch는 True가 "가린다"는 뜻
print(torch.allclose(y, y_torch, atol=1e-5))

In [ ]:
n_params = sum(p.numel() for p in mha.parameters())
print("파라미터 수:", n_params, "= 4 x (16 x 16 + 16)")